# CTRL00 — Memory Availability and Geometry Control

C01 showed a clean manipulation \(Z_{\rm full}=1,\;Z_{\rm cap}=0,\;Z_{\rm rescue}\approx0.97\),
but CAP32 had identical visible history across instances. C02 fixes that and tests whether
the *geometry/representation* of accessible memory changes the surviving \(U\) trace.

Conditions:

- `full_raw`: raw `LABEL -> CITY` block far back.
- `cap32`: only the last 32 tokens of an **instance-specific neutral tail**.
- `far_norm`: same bindings in normalized sentences at the same far position.
- `near_norm`: the same normalized binding card moved close to the continuation.

All memory-bearing conditions end in the same instance-specific neutral tail, so their
local suffix is identical within an instance. CAP32 retains no binding values but no longer
collapses all 60 instances onto the same input.

Primary contrasts:

\[
\text{availability}: cap32-full\_raw
\]

\[
\text{representation}: far\_norm-full\_raw
\]

\[
\boxed{\text{geometry}: near\_norm-far\_norm}
\]

The scorer always receives the same original raw prior document plus the generated
continuation. Rescue cards are generation-side only.


In [ ]:
from pathlib import Path
import gc, json, random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd
OUT = ROOT / "results" / "controls" / "CTRL00_memory_availability_geometry"
OUT.mkdir(parents=True, exist_ok=True)

GENERATOR_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SCORER_ID = "Qwen/Qwen3-0.6B-Base"

SEED = 20260901
N = 60
CAP = 32
WINDOWS = [8, 32, 128, 256]
TARGET_OFFSETS = [12,20,28,36,44,52,60,68]
CONDITIONS = ["full_raw","cap32","far_norm","near_norm"]

rng = np.random.default_rng(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

CITIES = np.array([
    "Austin","Boston","Chicago","Denver","Houston","Miami","Phoenix","Portland",
    "Raleigh","Seattle","Tucson","Atlanta","Detroit","Nashville","Orlando","Madison",
    "Buffalo","Fresno","Cleveland","Richmond","Boise","Spokane","Memphis","Baltimore",
])
ALPH = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ"))


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build controlled documents

In [2]:
PREAMBLE = """
This field manual describes routine logistics procedures used by distributed inspection
teams. Reports are written in chronological order and should remain internally consistent.
Each team reviews equipment condition, transport constraints, weather, communications,
access permissions, and the order of routine checks. Inspectors record observations
without embellishment and distinguish confirmed information from assumptions. When an
assignment is given earlier in a report, later sections should preserve that assignment
unless an explicit reassignment is recorded. Reports use concise technical prose and avoid
unnecessary restatement. Site visits involve arrival checks, an equipment survey,
verification of local constraints, documentation of anomalies, and a closing note
describing what should happen next. Communications are logged separately from observations
so later readers can reconstruct the sequence of decisions. Equipment identifiers,
assigned destinations, and status changes are treated as factual state that carries
forward through the report. Before departure, teams verify cases, radios, batteries,
weather covers, transport documents, and local access requirements. Routine inspection
details should not overwrite earlier operational assignments.
""".strip()

BRIDGE = """
After the assignments were recorded, the team completed an unrelated equipment inspection.
Battery housings were checked, cable connectors were reseated, protective cases were
counted, and backup power modules were verified. These routine checks did not modify any
earlier assignment.
""".strip()

def rlabel():
    return "".join(rng.choice(ALPH, size=5, replace=True).tolist())

def rcode(n):
    chars = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ23456789"))
    return "".join(rng.choice(chars, size=n, replace=True).tolist())

def make_tail():
    return (
        f"Administrative reference {rcode(7)} was logged for this visit. "
        f"Inventory listed {int(rng.integers(6,30))} seals, "
        f"{int(rng.integers(5,25))} weather covers, "
        f"{int(rng.integers(8,40))} numbered tags, and "
        f"{int(rng.integers(4,18))} protective cases. "
        f"The radio test code was {rcode(4)}. "
        "These bookkeeping details are unrelated to destination assignments and "
        "do not modify earlier operational state."
    )

def raw_block(b):
    return (
        "The following four assignments are now fixed:\n"
        + "\n".join(f"{k} -> {v}" for k,v in b.items())
        + "\nThese assignments remain in force for the rest of the report."
    )

def norm_block(b):
    return (
        "[COMPACT STATE]\n"
        + "\n".join(f"{k} is assigned to {v}." for k,v in b.items())
        + "\n[END COMPACT STATE]"
    )

design=[]
for i in range(N):
    labels=[]
    while len(labels)<4:
        x=rlabel()
        if x not in labels:
            labels.append(x)
    cities=rng.choice(CITIES,size=4,replace=False).tolist()
    b=dict(zip(labels,cities))
    raw,norm,tail=raw_block(b),norm_block(b),make_tail()
    original=PREAMBLE+"\n\n"+raw+"\n\n"+BRIDGE+"\n\n"+tail
    design.append({
        "instance_id":i,"bindings":b,"raw":raw,"norm":norm,
        "tail":tail,"original_prior":original
    })

display(pd.DataFrame([{
    "instance_id":d["instance_id"],
    "bindings":json.dumps(d["bindings"]),
    "tail":d["tail"][:80]+"..."
} for d in design[:5]]))


,instance_id,bindings,tail
0,0,"{""URVPU"": ""Baltimore"", ""JYGAD"": ""Memphis"", ""JA...",Administrative reference RHS9SGX was logged fo...
1,1,"{""ELGBL"": ""Orlando"", ""GHTWF"": ""Denver"", ""RDPKF...",Administrative reference Z3GJ5P9 was logged fo...
2,2,"{""RRDJF"": ""Houston"", ""LHTFC"": ""Detroit"", ""TEMD...",Administrative reference WLC7HJ4 was logged fo...
3,3,"{""ZCBAU"": ""Seattle"", ""HJVNA"": ""Raleigh"", ""WPQQ...",Administrative reference 9UKKFAW was logged fo...
4,4,"{""DQTUJ"": ""Orlando"", ""KEYBS"": ""Seattle"", ""XCKZ...",Administrative reference 8A9PL9H was logged fo...


## 2. Load generator and verify anti-degeneracy / geometry

In [3]:
gtok=AutoTokenizer.from_pretrained(GENERATOR_ID,trust_remote_code=True)
gmodel=AutoModelForCausalLM.from_pretrained(
    GENERATOR_ID,torch_dtype="auto",device_map="auto",trust_remote_code=True
)
gmodel.eval()

def enc(text):
    return gtok(text,add_special_tokens=False,return_tensors="pt").input_ids[0].cpu()

def last32(text):
    ids=enc(text)
    return gtok.decode(ids[-CAP:],skip_special_tokens=True)

def states(d):
    full=PREAMBLE+"\n\n"+d["raw"]+"\n\n"+BRIDGE+"\n\n"+d["tail"]
    far=PREAMBLE+"\n\n"+d["norm"]+"\n\n"+BRIDGE+"\n\n"+d["tail"]
    near=PREAMBLE+"\n\n"+BRIDGE+"\n\n"+d["norm"]+"\n\n"+d["tail"]
    cap=last32(full)
    return {"full_raw":full,"cap32":cap,"far_norm":far,"near_norm":near}

geom=[]
caps=[]
for d in design:
    s=states(d)
    fi,fai,ni=enc(s["full_raw"]),enc(s["far_norm"]),enc(s["near_norm"])
    assert torch.equal(fi[-CAP:],fai[-CAP:])
    assert torch.equal(fi[-CAP:],ni[-CAP:])
    caps.append(s["cap32"])
    geom.append({
        "instance_id":d["instance_id"],
        "full_tokens":len(fi),
        "far_memory_end_to_cont":len(enc(BRIDGE+"\n\n"+d["tail"])),
        "near_memory_end_to_cont":len(enc(d["tail"])),
        "tail_tokens":len(enc(d["tail"])),
    })

geom=pd.DataFrame(geom)
display(geom.describe().T)
print("unique CAP32 strings:",len(set(caps)),"/",N)
assert len(set(caps))==N
assert (geom["far_memory_end_to_cont"]>32).all()
assert (geom["near_memory_end_to_cont"]<geom["far_memory_end_to_cont"]).all()
print("geometry checks passed")


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 859.94it/s]


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
full_tokens,60.0,370.583333,1.871206,365.0,369.00,370.5,372.00,374.0
far_memory_end_to_cont,60.0,119.050000,1.477802,115.0,118.00,119.0,120.00,122.0
near_memory_end_to_cont,60.0,66.050000,1.477802,62.0,65.00,66.0,67.00,69.0
tail_tokens,60.0,66.050000,1.477802,62.0,65.00,66.0,67.00,69.0


unique CAP32 strings: 60 / 60
geometry checks passed


## 3. Coherent chat helpers

In [4]:
SYSTEM_USER=(
    "You are maintaining a field report. Treat the previous assistant message as the "
    "report written so far and answer or continue from it consistently."
)

def chat_ids(history,query):
    msgs=[
        {"role":"user","content":SYSTEM_USER},
        {"role":"assistant","content":history},
        {"role":"user","content":query},
    ]
    x=gtok.apply_chat_template(
        msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt"
    )
    return x.input_ids if hasattr(x,"input_ids") else x

def greedy(ids,max_new_tokens):
    ids=ids.to(gmodel.device)
    with torch.inference_mode():
        y=gmodel.generate(
            input_ids=ids,do_sample=False,max_new_tokens=max_new_tokens,
            pad_token_id=gtok.eos_token_id,eos_token_id=gtok.eos_token_id,use_cache=True
        )
    new=y[0,ids.shape[1]:]
    return gtok.decode(new,skip_special_tokens=True).strip(),new.detach().cpu()


## 4. Manipulation check — stop here first

Proceed only if `full_raw`, `far_norm`, and `near_norm` all retain high binding recall
while `cap32` remains near zero.


In [5]:
PROBE_PATH=OUT/"binding_probes.parquet"
existing=pd.read_parquet(PROBE_PATH) if PROBE_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"],existing["label"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for d in tqdm(design,desc="C02 probes"):
    ss=states(d)
    for cond in CONDITIONS:
        for lab,city in d["bindings"].items():
            key=(d["instance_id"],cond,lab)
            if key in done:
                continue
            q=f"What destination was assigned to {lab}? Reply with the destination city only."
            ans,ids=greedy(chat_ids(ss[cond],q),12)
            rows.append({
                "instance_id":d["instance_id"],"condition":cond,"label":lab,
                "expected_city":city,"answer":ans,
                "correct":city.lower() in ans.lower(),"answer_tokens":len(ids)
            })
    pd.DataFrame(rows).to_parquet(PROBE_PATH,index=False)

probes=pd.read_parquet(PROBE_PATH)
Z=probes.groupby(["instance_id","condition"])["correct"].mean().rename("Z_binding").reset_index()

display(Z.groupby("condition")["Z_binding"].agg(["mean","std","count"]))
display(probes[probes["instance_id"]==0][["condition","label","expected_city","answer","correct"]])


C02 probes: 100%|███████████████████████████████████████████████████████████████████████████████████████| 60/60 [02:26<00:00,  2.44s/it]


,mean,std,count
condition,,,
cap32,0.000000,0.000000,60
far_norm,0.950000,0.100844,60
full_raw,0.983333,0.062887,60
near_norm,0.950000,0.100844,60


,condition,label,expected_city,answer,correct
0,full_raw,URVPU,Baltimore,Baltimore,True
1,full_raw,JYGAD,Memphis,Memphis,True
2,full_raw,JAPEV,Buffalo,Buffalo,True
3,full_raw,LPBTZ,Nashville,Nashville,True
4,cap32,URVPU,Baltimore,URVPU was assigned to DFW (Dallas/Fort,False
5,cap32,JYGAD,Memphis,JYGAD was assigned to Guangzhou.,False
6,cap32,JAPEV,Buffalo,The destination for JAPEV is unspecified in th...,False
7,cap32,LPBTZ,Nashville,"I'm sorry, but I don't have enough information to",False
8,far_norm,URVPU,Baltimore,Baltimore,True
9,far_norm,JYGAD,Memphis,Memphis,True


## 5. Natural continuation branch

In [6]:
GEN_PATH=OUT/"natural_continuations.parquet"
QUERY=(
    "Continue the field report for about 150 to 190 words. Return to the previously "
    "assigned items and discuss their next operational steps consistently with the "
    "assignments already recorded. Integrate relevant locations naturally where useful. "
    "Do not reproduce the assignment table, do not use arrow notation, and do not "
    "describe this as a memory test. Write only the continuation."
)

existing=pd.read_parquet(GEN_PATH) if GEN_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for d in tqdm(design,desc="C02 continuations"):
    ss=states(d)
    for cond in CONDITIONS:
        key=(d["instance_id"],cond)
        if key in done:
            continue
        cont,ids=greedy(chat_ids(ss[cond],QUERY),240)
        rows.append({
            "instance_id":d["instance_id"],"condition":cond,
            "bindings_json":json.dumps(d["bindings"]),
            "original_prior":d["original_prior"],
            "continuation_text":cont,
            "scored_document":d["original_prior"]+"\n\n"+cont,
            "continuation_tokens":len(ids),
        })
    pd.DataFrame(rows).to_parquet(GEN_PATH,index=False)

gens=pd.read_parquet(GEN_PATH)
display(gens.groupby("condition")["continuation_tokens"].agg(["mean","std","count"]))
display(gens.groupby("condition")["continuation_text"].nunique().rename("n_unique").to_frame())


C02 continuations: 100%|████████████████████████████████████████████████████████████████████████████████| 60/60 [20:25<00:00, 20.42s/it]


,mean,std,count
condition,,,
cap32,216.983333,31.635901,60
far_norm,239.816667,1.420094,60
full_raw,238.333333,9.080929,60
near_norm,240.000000,0.000000,60


,n_unique
condition,
cap32,60
far_norm,60
full_raw,60
near_norm,60


## 6. Natural binding-use diagnostic

In [7]:
def natural_use(row):
    b=json.loads(row["bindings_json"])
    text=row["continuation_text"].lower()
    return float(np.mean([city.lower() in text for city in b.values()]))

gens["natural_binding_use"]=gens.apply(natural_use,axis=1)
gens.to_parquet(GEN_PATH,index=False)
display(gens.groupby("condition")["natural_binding_use"].agg(["mean","std","count"]))


,mean,std,count
condition,,,
cap32,0.000000,0.000000,60
far_norm,0.533333,0.407729,60
full_raw,0.479167,0.412443,60
near_norm,0.495833,0.432992,60


## 7. Load scorer and audit scorer-side raw-binding geometry

In [8]:
del gmodel
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

stok=AutoTokenizer.from_pretrained(SCORER_ID,trust_remote_code=True)
smodel=AutoModelForCausalLM.from_pretrained(
    SCORER_ID,torch_dtype="auto",device_map="auto",trust_remote_code=True
)
smodel.eval()
if stok.pad_token_id is None:
    stok.pad_token_id=stok.eos_token_id

rows=[]
for d in design:
    before=PREAMBLE+"\n\n"
    through=PREAMBLE+"\n\n"+d["raw"]
    prior=d["original_prior"]
    st=len(stok(before,add_special_tokens=False).input_ids)
    en=len(stok(through,add_special_tokens=False).input_ids)
    pl=len(stok(prior,add_special_tokens=False).input_ids)
    rows.append({
        "instance_id":d["instance_id"],"binding_start":st,"binding_end":en,"prior_len":pl,
        "binding_end_to_cont":pl-en,"binding_start_to_cont":pl-st
    })

scorer_geom=pd.DataFrame(rows)
display(scorer_geom.describe().T)
assert (scorer_geom["prior_len"]>=256).all()


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 310/310 [00:00<00:00, 1398.11it/s]


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
binding_start,60.0,207.000000,0.000000,207.0,207.00,207.0,207.00,207.0
binding_end,60.0,251.533333,1.096476,249.0,251.00,252.0,252.00,254.0
prior_len,60.0,370.583333,1.871206,365.0,369.00,370.5,372.00,374.0
binding_end_to_cont,60.0,119.050000,1.477802,115.0,118.00,119.0,120.00,122.0
binding_start_to_cont,60.0,163.583333,1.871206,158.0,162.00,163.5,165.00,167.0


## 8. Frozen context-utility scoring

In [9]:
SCORE_PATH=OUT/"scores.parquet"

def score_doc(document,prior):
    ids=stok(document,add_special_tokens=False).input_ids
    pids=stok(prior,add_special_tokens=False).input_ids
    start=len(pids)
    targets=[start+o for o in TARGET_OFFSETS if start+o<len(ids)]
    if len(targets)!=len(TARGET_OFFSETS):
        return None

    ctxs,tgts,ks=[],[],[]
    for t in targets:
        for k in WINDOWS:
            if t<k:
                return None
            ctxs.append(torch.tensor(ids[t-k:t],dtype=torch.long))
            tgts.append(ids[t])
            ks.append(k)

    L=max(map(len,ctxs))
    batch=torch.full((len(ctxs),L),stok.pad_token_id,dtype=torch.long)
    mask=torch.zeros_like(batch)
    for i,x in enumerate(ctxs):
        batch[i,-len(x):]=x
        mask[i,-len(x):]=1

    batch,mask=batch.to(smodel.device),mask.to(smodel.device)
    tgt=torch.tensor(tgts,dtype=torch.long,device=smodel.device)

    with torch.inference_mode():
        logits=smodel(input_ids=batch,attention_mask=mask,use_cache=False).logits[:,-1,:]
        lp=torch.log_softmax(logits.float(),dim=-1)
        losses=-lp[torch.arange(len(tgt),device=smodel.device),tgt]

    losses=losses.detach().cpu().numpy()
    out={}
    for k in WINDOWS:
        out[f"L_{k}"]=float(np.mean([v for v,kk in zip(losses,ks) if kk==k]))
    out["U_32"]=out["L_8"]-out["L_32"]
    out["U_128"]=out["L_8"]-out["L_128"]
    out["U_256"]=out["L_8"]-out["L_256"]
    out["U_long"]=.5*(out["U_128"]+out["U_256"])
    out["selectivity"]=out["U_long"]-out["U_32"]
    return out

existing=pd.read_parquet(SCORE_PATH) if SCORE_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for _,r in tqdm(gens.iterrows(),total=len(gens),desc="C02 scoring"):
    key=(int(r["instance_id"]),str(r["condition"]))
    if key in done:
        continue
    s=score_doc(r["scored_document"],r["original_prior"])
    rows.append({
        "instance_id":int(r["instance_id"]),"condition":r["condition"],
        "eligible":s is not None,**({} if s is None else s)
    })
    pd.DataFrame(rows).to_parquet(SCORE_PATH,index=False)

scores=pd.read_parquet(SCORE_PATH)
display(scores.groupby("condition")["eligible"].agg(["sum","count"]))


C02 scoring: 100%|████████████████████████████████████████████████████████████████████████████████████| 240/240 [03:34<00:00,  1.12it/s]


,sum,count
condition,,
cap32,60,60
far_norm,60,60
full_raw,60,60
near_norm,60,60


## 9. Joint paired analysis and planned contrasts

In [10]:
analysis=gens.merge(Z,on=["instance_id","condition"],how="left").merge(
    scores,on=["instance_id","condition"],how="left"
)
elig=analysis.pivot_table(
    index="instance_id",columns="condition",values="eligible",aggfunc="first"
).fillna(False)
joint=elig.index[elig[CONDITIONS].all(axis=1)]
a=analysis[analysis["instance_id"].isin(joint)].copy()

print("jointly eligible:",len(joint),"/",N)
display(a.groupby("condition")[[
    "Z_binding","natural_binding_use","U_32","U_128","U_256","U_long","selectivity"
]].agg(["mean","std","count"]))

def wide(metric):
    return a.pivot(index="instance_id",columns="condition",values=metric)[CONDITIONS].dropna()

def boot(metric,left,right,B=10000):
    w=wide(metric)
    d=(w[left]-w[right]).to_numpy(float)
    rr=np.random.default_rng(SEED)
    bs=np.empty(B)
    for i in range(B):
        ix=rr.integers(0,len(d),len(d))
        bs[i]=d[ix].mean()
    return {
        "metric":metric,"contrast":f"{left} - {right}","n":len(d),
        "estimate":float(d.mean()),
        "ci_low":float(np.quantile(bs,.025)),
        "ci_high":float(np.quantile(bs,.975)),
    }

planned=[
    ("cap32","full_raw"),
    ("far_norm","full_raw"),
    ("near_norm","far_norm"),
    ("far_norm","cap32"),
    ("near_norm","cap32"),
    ("near_norm","full_raw"),
]
metrics=["Z_binding","natural_binding_use","U_32","U_128","U_256","U_long","selectivity"]

result=pd.DataFrame([boot(m,l,r) for m in metrics for l,r in planned])
display(result[result["metric"].isin(["Z_binding","U_long","selectivity"])])
result.to_csv(OUT/"paired_contrasts.csv",index=False)


jointly eligible: 60 / 60


Z_binding                 natural_binding_use                  \
               mean       std count                mean       std count   
condition                                                                 
cap32      0.000000  0.000000    60            0.000000  0.000000    60   
far_norm   0.950000  0.100844    60            0.533333  0.407729    60   
full_raw   0.983333  0.062887    60            0.479167  0.412443    60   
near_norm  0.950000  0.100844    60            0.495833  0.432992    60   

               U_32                     U_128  ...           U_256            \
               mean       std count      mean  ... count      mean       std   
condition                                      ...                             
cap32      0.867801  0.629619    60  1.558325  ...    60  1.502218  0.620429   
far_norm   1.056200  0.554973    60  1.861683  ...    60  2.268745  1.093635   
full_raw   1.091986  0.620170    60  1.263234  ...    60  1.873076  1.014977   
near_norm  0.928553  0.585689    60  1.434288  ...    60  2.288176  0.959734   

                   U_long                 selectivity                  
          count      mean       std count        mean       std count  
condition                                                              
cap32        60  1.530271  0.606209    60    0.662471  0.580629    60  
far_norm     60  2.065214  1.001001    60    1.009014  0.916535    60  
full_raw     60  1.568155  0.797917    60    0.476169  0.480672    60  
near_norm    60  1.861232  0.757105    60    0.932679  0.676192    60  

[4 rows x 21 columns]

,metric,contrast,n,estimate,ci_low,ci_high
0,Z_binding,cap32 - full_raw,60,-0.983333,-0.995833,-0.966667
1,Z_binding,far_norm - full_raw,60,-0.033333,-0.058333,-0.012500
2,Z_binding,near_norm - far_norm,60,0.000000,-0.012500,0.012500
3,Z_binding,far_norm - cap32,60,0.950000,0.925000,0.975000
4,Z_binding,near_norm - cap32,60,0.950000,0.925000,0.975000
5,Z_binding,near_norm - full_raw,60,-0.033333,-0.058333,-0.012500
30,U_long,cap32 - full_raw,60,-0.037884,-0.282079,0.204857
31,U_long,far_norm - full_raw,60,0.497059,0.170567,0.822645
32,U_long,near_norm - far_norm,60,-0.203982,-0.501872,0.086747
33,U_long,far_norm - cap32,60,0.534942,0.200607,0.874195


# Interpretation

- `cap32 - full_raw`: removal of binding availability.
- `far_norm - full_raw`: representation change at matched far-memory position.
- `near_norm - far_norm`: **memory geometry** with the same normalized binding state.
- `far_norm/near_norm - cap32`: semantic rescue.

The strongest geometry evidence would be:

\[
Z_{\rm near\_norm}\approx Z_{\rm far\_norm}
\]

but

\[
U_{\rm near\_norm}\neq U_{\rm far\_norm}.
\]

That would show that the same accessible semantic state can leave different completed-text
traces depending on where it is supplied in the generator's history.

C02 is synthetic and exploratory with respect to the geometry direction.
